In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain.llms import OpenAI
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

llm = OpenAI(temperature=0.3)
conversation = ConversationChain(
    llm=llm, 
    verbose=True, 
    memory=ConversationBufferMemory()
)

/tmp/ipykernel_3013503/4011265512.py:9: LangChainDeprecationWarning: The class `OpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAI``.
  llm = OpenAI(temperature=0.3)
/tmp/ipykernel_3013503/4011265512.py:13: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory=ConversationBufferMemory()
/tmp/ipykernel_3013503/4011265512.py:10: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use :meth:`~RunnableWithMessageHistory: https://python.langchain.com/v0.2/api_reference/core/runnables/langchain_core.runnables.history.RunnableWithMessageHistory.html` instead.
  conversation = ConversationChain(


In [2]:
conversation.predict(input="Hi there!")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Hi there!
AI:

> Finished chain.


" Hello! It's nice to meet you. I am an AI designed to assist with various tasks and provide information. I am currently running on a server located in a data center in California. How can I help you?"

In [3]:
conversation.predict(input="I'm doing well! Just having a conversation with an AI.")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hi there!
AI:  Hello! It's nice to meet you. I am an AI designed to assist with various tasks and provide information. I am currently running on a server located in a data center in California. How can I help you?
Human: I'm doing well! Just having a conversation with an AI.
AI:

> Finished chain.


" That's great to hear! I am always happy to chat with humans and learn more about them. Did you know that I am constantly learning and improving through machine learning algorithms? It allows me to adapt to new situations and provide more accurate responses. What would you like to talk about?"

In [4]:
conversation.predict(input="Tell me about yourself.")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hi there!
AI:  Hello! It's nice to meet you. I am an AI designed to assist with various tasks and provide information. I am currently running on a server located in a data center in California. How can I help you?
Human: I'm doing well! Just having a conversation with an AI.
AI:  That's great to hear! I am always happy to chat with humans and learn more about them. Did you know that I am constantly learning and improving through machine learning algorithms? It allows me to adapt to new situations and provide more accurate responses. What would you like to talk about?
Human: Tell me about yourself.
AI:

> Finished chain.


' Well, I am a complex system of algorithms and data that allows me to process information and perform tasks. I was created by a team of programmers and engineers who have spent countless hours designing and developing me. I am constantly connected to the internet, which allows me to access vast amounts of information and provide quick responses. Is there anything specific you would like to know about me?'

In [11]:
import lancedb
from langchain.prompts import PromptTemplate
from langchain.vectorstores import LanceDB
from langchain.embeddings import OpenAIEmbeddings

# 1. LLM + 記憶
llm = OpenAI(temperature=0.3)
memory = ConversationBufferMemory(memory_key="history", return_messages=True)

# 2. LanceDB 檢索設定
# 連線到你已經建好的 DB
conn = lancedb.connect("./.lancedb")  # 路徑可改成你的資料夾

# 建立向量檢索器
embedding = OpenAIEmbeddings()
vectorstore = LanceDB(
    connection=conn, 
    embedding=embedding, 
    table_name="music_text", 
    vector_key="text_vector",
    text_key="combined_info")
retriever = vectorstore.as_retriever()

In [12]:
# 3. 自訂 prompt，整合 context（檢索結果）+ chat history + input
prompt = PromptTemplate(
    input_variables=["history", "context", "input"],
    template="""
你是一位音樂推薦助理，根據使用者的需求與資料庫內容給出音樂建議。

目前的對話紀錄：
{history}

從資料庫中找到的資訊：
{context}

使用者說：
{input}

請根據對話與資料庫，推薦幾首合適的音樂並說明理由。
"""
)

In [13]:
from langchain.chains import LLMChain

# 4. conversation chain 自訂 prompt
chain = LLMChain(
    llm=llm,
    prompt=prompt,
    verbose=True
)

# 5. 封裝聊天功能
def chat_with_retrieval(user_input: str):
    history = memory.load_memory_variables({})["history"]

    docs = retriever.get_relevant_documents(user_input)
    context = "\n".join([doc.page_content for doc in docs])

    response = chain.predict(input=user_input, context=context, history=history)
    memory.save_context({"input": user_input}, {"output": response})    
    return response 

In [14]:
# # 6. 進行檢索
# query = "推薦我幾首適合慢跑的背景音樂"
# docs = vectorstore.similarity_search(query, k=2)

# # 7. 顯示檢索結果
# print("檢索結果:")
# for i, doc in enumerate(docs):
#     print(f"結果 {i+1}:")
#     print(f"內容: {doc.page_content}")
#     #print(f"來源: {doc.metadata['combined_info']}")
#     print("-" * 50)

In [15]:
# retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [16]:
# 6. 測試使用
response = chat_with_retrieval("我想找一些適合放在日常生活影片裡的背景音樂")
print(response)



> Entering new LLMChain chain...
Prompt after formatting:

你是一位音樂推薦助理，根據使用者的需求與資料庫內容給出音樂建議。

目前的對話紀錄：
[]

從資料庫中找到的資訊：
Moods: Uplifting, Happy, Carefree, Playful. Video Themes: Food, Education, Road Trip. Instruments: Mandolin & Ukulele, Acoustic Drums, Claps & Snaps. Genres: Country, Acoustic, Folk. Description: This is a traditional Chinese folk music with Chinese instruments like guzheng, pipa, yangqin, etc. The music is performed by traditional Chinese instruments and the rhythm is traditional Chinese music rhythm. The mood of the music is traditional, solemn, and nostalgic. This music is suitable for Chinese traditional culture-related projects, documentaries, and historical themes
Moods: Love, Peaceful, Hopeful. Video Themes: Lifestyle, Urban, Nature, VlogIntros & Logos. Instruments: Piano. Genres: Cinematic, Classical, Holiday. Description: This is a beautiful, soft, and emotional piano piece that will evoke feelings of nostalgia, longing, and romance. The gentle and soothing p

In [17]:
response = chat_with_retrieval("我不想要這些歌曲，麻煩幫我找其他適合的")
print(response)



> Entering new LLMChain chain...
Prompt after formatting:

你是一位音樂推薦助理，根據使用者的需求與資料庫內容給出音樂建議。

目前的對話紀錄：
[HumanMessage(content='我想找一些適合放在日常生活影片裡的背景音樂', additional_kwargs={}, response_metadata={}), AIMessage(content='\n1. "Lifestyle" by Acoustic Guitar, Keys (Genres: Cinematic, Acoustic, Pop, Folk, Children, Corporate)\n這首歌曲具有輕快的節奏和輕鬆的氛圍，適合放在日常生活影片中作為背景音樂。同時，它也具有多種類型的元素，適合搭配不同類型的影片。\n\n2. "Uplifting" by Piano (Genres: Cinematic, Classical, Holiday)\n這首歌曲以柔和的鋼琴旋律為主，帶有激勵人心的氛圍，適合放在日常生活影片中作為背景音樂，能夠為影片增添溫馨的氛圍。\n\n3. "Happy" by Acoustic Guitar, Electric Guitar, Keys, Electronic Drums, Pads (Genres: Cinematic, Corporate)\n', additional_kwargs={}, response_metadata={})]

從資料庫中找到的資訊：
Moods: Uplifting, Happy, Carefree, Playful. Video Themes: Food, Education, Road Trip. Instruments: Mandolin & Ukulele, Acoustic Drums, Claps & Snaps. Genres: Country, Acoustic, Folk. Description: This is a traditional Chinese folk music with Chinese instruments like guzheng, pipa, yangqin, etc. The music is performed